## **Semana W08 – Periodic Table of AI** ✨

**Descripción:** 14 categorías de empresas de IA fundadas/financiadas entre febrero de 2025 y febrero de 2026

**Dataset:** https://data.world/makeovermonday/2026wk-8-the-periodic-table-of-ai-startups  
**Fuente:** https://www.voronoiapp.com/investing/The-Periodic-Table-of-AI-Startups--14-categories-of-AI-companies-foundedfunded-Feb-2025-Feb-2026-7663  

**Fecha de publicación:** 2026-02-23

## **Librerías**

In [1]:
import pandas as pd

import plotly.express as px

In [2]:
GROUP_COLORS = {
    'foundation': '#1f40a6',
    'horizontal': '#00bf63',
    'vertical': '#ff751f',
    'emerging': '#bf0017'
}

TREND_COLORS = {
    'Explosive': '#FF6B35',
    'Rising': '#4ECDC4',
    'Steady': '#F7B801',
    'Mature': '#95389E'
}

In [3]:
RAW_PATH = '../data/raw/data_raw.xlsx'
PROCESSED_PATH = '../data/processed/ai_periodic_clean.parquet'

### **1. Contexto y preguntas iniciales** 📋

**Historia del dataset:**  
305 startups de IA que recibieron financiamiento entre Feb 2025 - Feb 2026, agrupadas en 14 categorías que representan diferentes capas del ecosistema (Foundation, Horizontal, Vertical, Emerging & Frontier).

**Visualización original:**  
Tabla periódica estilizada donde cada categoría es una "tarjeta" con símbolo químico, funding total, número de startups, crecimiento YoY y tendencia de momentum.

**Preguntas clave:**
1. ¿Qué categorías concentran más capital vs. más startups? ¿Hay desequilibrios?
2. ¿Cómo se relacionan funding, startup_count y yoy_growth_pct entre sí?
3. ¿Qué patrones emergen al cruzar trend (Explosive/Rising/Steady/Mature) con las métricas numéricas?
4. ¿Dónde está la mayor oportunidad de inversión (alto crecimiento + bajo capital actual)?

### **2. Exploración inicial** 🔍

In [4]:
raw_df = pd.read_excel(RAW_PATH)
print(f"Filas: {len(raw_df)}, Columnas: {len(raw_df.columns)}")
print(f"\nColumnas: {list(raw_df.columns)}")

Filas: 14, Columnas: 11

Columnas: ['id', 'symbol', 'name', 'group', 'group_label', 'funding_billions', 'startup_count', 'yoy_growth_pct', 'trend', 'examples', 'factoid']


In [5]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                14 non-null     int64  
 1   symbol            14 non-null     object 
 2   name              14 non-null     object 
 3   group             14 non-null     object 
 4   group_label       14 non-null     object 
 5   funding_billions  14 non-null     float64
 6   startup_count     14 non-null     int64  
 7   yoy_growth_pct    14 non-null     int64  
 8   trend             14 non-null     object 
 9   examples          14 non-null     object 
 10  factoid           14 non-null     object 
dtypes: float64(1), int64(3), object(7)
memory usage: 1.3+ KB


In [6]:
raw_df.head()

,id,symbol,name,group,group_label,funding_billions,startup_count,yoy_growth_pct,trend,examples,factoid
0,1,Fm,Foundation Models & LLMs,foundation,Foundation Layer,80.0,15,75,Mature,"OpenAI, Anthropic, xAI, Mistral, DeepSeek",OpenAI alone raised $40B in a single round
1,2,If,AI Infrastructure,foundation,Foundation Layer,7.0,28,100,Steady,"Databricks, Baseten, Groq, Upscale AI, Scale AI","Chips, MLOps, cloud & compute — the picks & sh..."
2,3,Ag,AI Agents & Copilots,horizontal,Horizontal AI,13.0,48,260,Explosive,"Sierra, Cursor, Cognition, Decagon, Lovable",Market tripled
3,4,En,Enterprise AI Software,horizontal,Horizontal AI,8.0,42,150,Rising,"Glean, Uniphore, Writer, Genspark, Distyl AI","Departmental AI spending hit $7.3B, up 4.1× YoY"
4,5,Cd,AI Coding & Dev Tools,horizontal,Horizontal AI,4.8,22,320,Explosive,"Cursor, Lovable, Replit, Devin, Reflection AI",Fastest growing — 'vibe coding' is a new paradigm


In [7]:
raw_df.describe()

,id,funding_billions,startup_count,yoy_growth_pct
count,14.0000,14.000000,14.000000,14.000000
mean,7.5000,9.821429,21.785714,177.500000
std,4.1833,20.446673,11.891681,66.701113
min,1.0000,1.200000,10.000000,75.000000
25%,4.2500,2.275000,14.250000,132.500000
50%,7.5000,3.750000,17.000000,175.000000
75%,10.7500,6.500000,26.500000,200.000000
max,14.0000,80.000000,48.000000,320.000000


In [8]:
print("Valores únicos por columna categórica:")
print(f"  - group: {raw_df['group'].unique()}")
print(f"  - group_label: {raw_df['group_label'].unique()}")
print(f"  - trend: {raw_df['trend'].unique()}")

Valores únicos por columna categórica:
  - group: ['foundation' 'horizontal' 'vertical' 'emerging']
  - group_label: ['Foundation Layer' 'Horizontal AI' 'Vertical AI' 'Emerging & Frontier']
  - trend: ['Mature' 'Steady' 'Explosive' 'Rising']


#### **Análisis de correlaciones entre métricas numéricas**

In [9]:
numeric_cols = ['funding_billions', 'startup_count', 'yoy_growth_pct']
corr_matrix = raw_df[numeric_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    aspect='equal',
    title='Correlaciones entre métricas numéricas'
)
fig.update_layout(
    width=650,
    height=550,
    coloraxis_colorbar=dict(title='Correlación'),
    font=dict(size=12),
    margin=dict(l=60, r=20, t=80, b=40)
)
fig.show()

#### **Distribución de trend y su relación con métricas**

In [10]:
trend_summary = raw_df.groupby('trend')[numeric_cols].agg(['mean', 'median', 'std'])
trend_summary

funding_billions                  startup_count                    \
                      mean median       std          mean median        std   
trend                                                                         
Explosive         6.333333    4.8  6.047589     28.333333   22.0  17.387735   
Mature           80.000000   80.0       NaN     15.000000   15.0        NaN   
Rising            3.828571    3.5  2.203569     21.000000   16.0  12.069245   
Steady            3.900000    2.5  2.688866     19.333333   20.0   9.018500   

          yoy_growth_pct                    
                    mean median        std  
trend                                       
Explosive     276.666667  260.0  37.859389  
Mature         75.000000   75.0        NaN  
Rising        175.714286  180.0  23.704530  
Steady        116.666667  120.0  15.275252

### **3. Limpieza y transformación** 🧼

In [11]:
processed_df = raw_df.copy()

processed_df['funding_pct_total'] = (
    processed_df['funding_billions'] / processed_df['funding_billions'].sum() * 100
)
processed_df['funding_per_startup'] = (
    processed_df['funding_billions'] / processed_df['startup_count']
)
processed_df['rank_by_funding'] = (
    processed_df['funding_billions'].rank(ascending=False, method='min').astype(int)
)
processed_df['rank_by_yoy'] = (
    processed_df['yoy_growth_pct'].rank(ascending=False, method='min').astype(int)
)

processed_df[['name', 'funding_billions', 'funding_pct_total', 
              'funding_per_startup', 'rank_by_funding', 'rank_by_yoy']].head()

,name,funding_billions,funding_pct_total,funding_per_startup,rank_by_funding,rank_by_yoy
0,Foundation Models & LLMs,80.0,58.181818,5.333333,1,14
1,AI Infrastructure,7.0,5.090909,0.250000,4,13
2,AI Agents & Copilots,13.0,9.454545,0.270833,2,2
3,Enterprise AI Software,8.0,5.818182,0.190476,3,9
4,AI Coding & Dev Tools,4.8,3.490909,0.218182,6,1


In [12]:
processed_df.to_parquet(PROCESSED_PATH, index=False)
print(f"✓ Dataset procesado guardado en: {PROCESSED_PATH}")

✓ Dataset procesado guardado en: ../data/processed/ai_periodic_clean.parquet


### **4. Visualizaciones** 📊

#### **4.1 Scatter: Funding vs Startups (tamaño = YoY, color = Trend)**

In [13]:
from src.charts.comparison import plot_scatter_with_size

fig = plot_scatter_with_size(
    processed_df,
    x_col='startup_count',
    y_col='funding_billions',
    size_col='yoy_growth_pct',
    color_col='trend',
    text_col='symbol',
    color_map=TREND_COLORS,
    title='Relación Funding vs Startups por Categoría (Tamaño = YoY Growth %)',
    x_label='Número de Startups',
    y_label='Funding (Billones USD)',
    theme='light',
    transparent_bg=True
)
fig.show()

#### 4.2 Boxplot: YoY Growth por Trend

In [14]:
from src.charts.distribution import plot_category_boxplot

trend_order = ['Explosive', 'Rising', 'Steady', 'Mature']

fig = plot_category_boxplot(
    processed_df,
    category_col='trend',
    value_col='yoy_growth_pct',
    category_order=trend_order,
    color_map=TREND_COLORS,
    title='Distribución de Crecimiento YoY por Tendencia',
    x_label='Trend',
    y_label='YoY Growth (%)'
)
fig.show()

#### 4.3 Treemap: Funding por Trend

In [15]:
from src.charts.composition import plot_hierarchical_treemap

trend_funding = (
    processed_df.groupby('trend', as_index=False)['funding_billions']
    .sum()
    .sort_values('funding_billions', ascending=False)
)

fig = plot_hierarchical_treemap(
    trend_funding,
    path_cols=['trend'],
    value_col='funding_billions',
    color_col='trend',
    color_map=TREND_COLORS,
    title='Distribución de Funding Total por Tendencia'
)
fig.show()

In [16]:
# from src.charts.composition import build_periodic_table_figure

# fig = build_periodic_table_figure(
#     processed_df,
#     group_colors=GROUP_COLORS,
#     trend_colors=TREND_COLORS
# )
# fig.show()

In [17]:
# from src.charts.composition import build_periodic_table_figure

# fig = build_periodic_table_figure(processed_df)
# fig.show()

#### 4.5 Cards periódicas por registro
Generamos una tarjeta por categoría utilizando `build_periodic_card`, respetando la paleta definida en `CARD_STYLE` y `TREND_COLORS`.

In [ ]:
from IPython.display import display
from pathlib import Path
import plotly.io as pio
from src.charts.composition import build_periodic_card

assets_dir = Path('../assets')
assets_dir.mkdir(exist_ok=True)

pio.kaleido.scope.default_format = 'png'
pio.kaleido.scope.default_width = 320
pio.kaleido.scope.default_height = 320
pio.kaleido.scope.default_scale = 1

cards = []
total_cards = len(processed_df)
preview_limit = 2  # Muestra solo las primeras N tarjetas como referencia

for idx, (_, row) in enumerate(processed_df.iterrows(), start=1):
    symbol = row.get('symbol', 'unknown').lower()
    group_color = GROUP_COLORS.get(row['group'], '#2F4AA0')
    dot_color = TREND_COLORS.get(row['trend'], '#D86BC7')

    fig = build_periodic_card(
        row,
        width=320,
        height=320,
        header_color=group_color,
        border_color=group_color,
        dot_color=dot_color
    )
    cards.append(fig)

    output_path = assets_dir / f'card_{symbol}.png'
    img_bytes = fig.to_image(format='png', width=320, height=320, scale=1)
    output_path.write_bytes(img_bytes)

    if idx <= preview_limit:
        display(fig)

    print(f"✓ [{idx}/{total_cards}] Guardado {output_path.name}")

print(f"\n✓ {len(cards)} tarjetas exportadas a {assets_dir.resolve()}")

### **5. Conclusiones y próximos pasos** 🚀

**Insights principales:**

1. **Concentración extrema de capital**: Foundation Models captura $80B (>50% del total) con solo 15 startups, mientras categorías "Explosive" como AI Coding tienen $4.8B repartidos en 22 startups.

2. **Correlaciones identificadas**:
   - Funding y startup_count muestran correlación positiva moderada.
   - YoY growth NO correlaciona fuertemente con funding actual, sugiriendo que las categorías emergentes (alto crecimiento) aún no concentran capital masivo.

3. **Patrones por Trend**:
   - **Explosive**: Alto YoY (promedio ~260%), funding moderado, pocas startups → oportunidad de inversión temprana.
   - **Mature**: Bajo YoY (~75%), funding masivo, pocas startups → mercado consolidado.
   - **Rising/Steady**: Zona intermedia con balance entre crecimiento y capital.

4. **Desequilibrios funding/startups**: Foundation Models tiene $5.3B por startup vs AI Coding con $0.22B por startup, revelando diferentes etapas de madurez y estrategias de financiamiento.

**Mejoras en la visualización**:
- Reducción del tamaño del símbolo químico para priorizar métricas clave (funding, startups, YoY).
- Uso de íconos de tendencia y colores diferenciados por macrogrupo.
- Visualizaciones complementarias (scatter, boxplot, treemap) que revelan correlaciones y patrones no evidentes en la tabla original.

**Próximos pasos**:
- Publicar visualización en Tableau Public o plataforma similar.
- Compartir en redes con #MakeoverMonday #DataViz.
- Explorar datos históricos si están disponibles para analizar evolución temporal de categorías.

### **6. Conclusiones y próximos pasos** 🚀
- Insight principal 1
- Insight principal 2
- Ideas para iteraciones futuras.